# 🎬 Sistema de Recomendación con LightFM
Modelo híbrido de factorización matricial (géneros, sinopsis, imagen) sobre MovieLens 1M.



## 1. Instalación de dependencias

In [1]:
!pip install scikit-learn scipy pandas numpy matplotlib seaborn tqdm
!pip install git+https://github.com/daviddavo/lightfm
!pip install recommenders

  Cloning https://github.com/daviddavo/lightfm to /tmp/pip-req-build-4vipp0ok
  Running command git clone --filter=blob:none --quiet https://github.com/daviddavo/lightfm /tmp/pip-req-build-4vipp0ok
  Resolved https://github.com/daviddavo/lightfm to commit f0eb500ead54ab65eb8e1b3890337a7223a35114
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp312-cp312-linux_x86_64.whl size=1099138 sha256=0cd797eeed29b5739a3a79800545297850ac7b6541a93a13d51759ac73ff2fdb
  Stored in directory: /tmp/pip-ephem-wheel-cache-zasvoy5j/wheels/fd/89/93/70c1e5f378ee5043de89387ee3ef6852ff39e3b9eb44ecc1a3
Successfully built lightfm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.3/355.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

## 2. Carga de datos y artefactos

In [2]:
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import load_npz, hstack, eye, csr_matrix
import kagglehub

kaggle_path = kagglehub.dataset_download("odedgolden/movielens-1m-dataset")

ratings = pd.read_csv(f"{kaggle_path}/ratings.dat", sep="::", engine="python",
                      names=["user_id", "movie_id", "rating", "timestamp"])
movies  = pd.read_csv(f"{kaggle_path}/movies.dat", sep="::", engine="python",
                      names=["movie_id", "title", "genres"], encoding="latin-1")

with open("encoders/le_user.pkl", "rb") as f:
    le_user = pickle.load(f)
with open("encoders/le_item.pkl", "rb") as f:
    le_item = pickle.load(f)

genre_features = load_npz("encoders/genre_features.npz")
item_table     = pd.read_csv("item_table.csv")

text_emb  = np.load("embeddings/text_embeddings.npy")
img_emb   = np.load("embeddings/image_embeddings.npy")
fused_emb = np.load("embeddings/fused_embeddings.npy")

ratings['user_idx'] = le_user.transform(ratings['user_id'])
ratings['item_idx'] = le_item.transform(ratings['movie_id'])
n_users = len(le_user.classes_)
n_items = len(le_item.classes_)
item_ids_ordered = le_item.classes_

interaction_matrix = csr_matrix(
    (ratings['rating'].values,
     (ratings['user_idx'].values, ratings['item_idx'].values)),
    shape=(n_users, n_items)
)
print(f"Usuarios: {n_users}  Items: {n_items}  Interacciones: {interaction_matrix.nnz}")

100%|██████████| 5.83M/5.83M [00:00<00:00, 64.3MB/s]

Extracting files...


Usuarios: 6040  Items: 3706  Interacciones: 1000209


## 3. Construcción de matrices de features

In [3]:
it = item_table.set_index('movie_id')

def build_emb_matrix(emb_array, dim):
    mat = np.zeros((n_items, dim))
    for item_idx, movie_id in enumerate(item_ids_ordered):
        if movie_id in it.index:
            emb_idx = it.loc[movie_id, 'emb_idx']
            mat[item_idx] = emb_array[emb_idx]
    return mat

text_mat  = build_emb_matrix(text_emb,  512)
fused_mat = build_emb_matrix(fused_emb, 512)
image_mat = build_emb_matrix(img_emb, 512)

item_feat_base  = genre_features
item_feat_text  = hstack([genre_features, csr_matrix(text_mat)])
item_feat_image = hstack([genre_features, csr_matrix(image_mat)])
item_feat_full  = hstack([genre_features, csr_matrix(fused_mat)])

user_feat = eye(n_users, format='csr')

print(f"item_feat_base:  {item_feat_base.shape}")
print(f"item_feat_text:  {item_feat_text.shape}")
print(f"item_feat_image: {item_feat_image.shape}")
print(f"item_feat_full:  {item_feat_full.shape}")

item_feat_base:  (3706, 18)
item_feat_text:  (3706, 530)
item_feat_image: (3706, 530)
item_feat_full:  (3706, 530)


## 4. Entrenamiento
Pérdida WARP, 30 épocas, 64 componentes latentes.

In [4]:
from lightfm import LightFM
from lightfm.cross_validation import random_train_test_split
from tqdm import tqdm

train, test = random_train_test_split(interaction_matrix,
                                      test_percentage=0.2, random_state=42)

train_csr  = train.tocsr()
train_seen = {u: set(train_csr[u].indices) for u in range(n_users)}

In [6]:
items = pd.read_csv("item_table.csv")

idx_to_title  = items.set_index("emb_idx")["clean_title"].to_dict()
idx_to_year   = items.set_index("emb_idx")["year"].to_dict()
idx_to_genres = items.set_index("emb_idx")["genres"].to_dict()

In [7]:
user_id = 9
train_csr = train.tocsr()

print(f"\n=== Películas vistas en TRAIN por el usuario {user_id} ===")
print(f"{'Título':<45} {'Año':<6} {'Géneros':<35} {'Rating'}")
print("-"*100)

fila = train_csr[user_id]

for item, rating in zip(fila.indices, fila.data):
    print(f"{idx_to_title.get(item, f'item_{item}'):<45} "
          f"{idx_to_year.get(item, '?'):<6} "
          f"{idx_to_genres.get(item, '?'):<35} "
          f"{rating}")


=== Películas vistas en TRAIN por el usuario 9 ===
Título                                        Año    Géneros                             Rating
----------------------------------------------------------------------------------------------------
Jumanji                                       1995   Adventure|Children's|Fantasy        5
Powder                                        1995   Drama|Sci-Fi                        3
Twelve Monkeys                                1995   Drama|Sci-Fi                        5
Pocahontas                                    1995   Animation|Children's|Musical|Romance 4
Before and After                              1996   Drama|Mystery                       3
Amazing Panda Adventure, The                  1995   Adventure|Children's                5
Living in Oblivion                            1995   Comedy                              2
Moonlight and Valentino                       1995   Drama|Romance                       3
Umbrellas of Cherbourg

In [6]:
def train_lightfm(item_features, label, epochs=30):
    model = LightFM(no_components=64, loss='warp', learning_rate=0.05,
                    item_alpha=1e-6, user_alpha=1e-6, random_state=42)
    with tqdm(total=epochs, desc=f"[LightFM] {label}", unit="epoch") as pbar:
        for epoch in range(epochs):
            model.fit_partial(train, user_features=user_feat,
                              item_features=item_features,
                              epochs=1, num_threads=4)
            pbar.update(1)
    return model

lfm_base = train_lightfm(item_feat_base, "base (generos)")
lfm_text = train_lightfm(item_feat_text, "base + sinopsis")
lfm_image = train_lightfm(item_feat_image, "base + imagen")
lfm_full = train_lightfm(item_feat_full, "base + sinopsis + imagen")

[LightFM] base + sinopsis + imagen: 100%|██████████| 30/30 [2:07:08<00:00, 254.27s/epoch]


## 5. Evaluación — Precision@10 y Recall@10

In [8]:
from lightfm.evaluation import precision_at_k, recall_at_k

def eval_accuracy(model, item_features, label):
    p10 = precision_at_k(model, test, k=10,
                         user_features=user_feat,
                         item_features=item_features).mean()
    r10 = recall_at_k(model, test, k=10,
                      user_features=user_feat,
                      item_features=item_features).mean()
    print(f"[LightFM] {label:30s}  P@10={p10:.4f}  R@10={r10:.4f}")
    return p10, r10

p_base, r_base = eval_accuracy(lfm_base, item_feat_base, "base (generos)")
p_text, r_text = eval_accuracy(lfm_text, item_feat_text, "base + sinopsis")
p_img, r_img = eval_accuracy(lfm_image, item_feat_image, "base + imagen")
p_full, r_full = eval_accuracy(lfm_full, item_feat_full, "base + sinopsis + imagen")

[LightFM] base (generos)                  P@10=0.0456  R@10=0.0179
[LightFM] base + sinopsis                 P@10=0.0911  R@10=0.0464
[LightFM] base + imagen                   P@10=0.0856  R@10=0.0428
[LightFM] base + sinopsis + imagen        P@10=0.0823  R@10=0.0401


## 6. Métricas de Diversidad y Novedad

Se implementan las tres métricas según las definiciones del curso:
Cita IA: https://claude.ai/share/f19c8cb6-6882-44af-8b06-cdf672f7f409


In [9]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

# Popularidad de cada item
item_counts     = np.bincount(ratings['item_idx'].values, minlength=n_items).astype(float)
item_popularity = item_counts / item_counts.sum()   # pop_i ∈ (0, 1)

# ILS Intra-List Similarity (Ziegler)
def ils_ziegler(top_k_lists, item_emb_matrix):
    scores = []
    for user, items in top_k_lists.items():
        if len(items) < 2:
            continue
        vecs = item_emb_matrix[items]
        sim  = cosine_similarity(vecs)
        n    = len(items)
        total = (sim.sum() - np.trace(sim)) / 2
        scores.append(total)
    return np.mean(scores)

# Novedad Self-information
def novedad_self_info(top_k_lists, item_popularity):
    user_novs = []
    for user, items in top_k_lists.items():
        pops = item_popularity[items]
        pops = np.clip(pops, 1e-10, None)
        nov  = np.mean(np.log(1.0 / pops))
        user_novs.append(nov)
    return np.mean(user_novs)

# Diversidad temporal (Lathia)
def diversidad_lathia(top_k_lists, train_seen, k=10):
    scores = []
    for user, l2_items in top_k_lists.items():
        l1 = train_seen.get(user, set())
        l2 = set(l2_items)
        nuevos = l2 - l1
        scores.append(len(nuevos) / k)
    return np.mean(scores)

print("Funciones de métricas definidas.")
print("  - ILS (Ziegler): similitud intra-lista coseno")
print("  - Novedad (Self-information): log(1/pop) promedio")
print("  - Diversidad Lathia: fracción de items nuevos respecto al histórico")


Funciones de métricas definidas.
  - ILS (Ziegler): similitud intra-lista coseno
  - Novedad (Self-information): log(1/pop) promedio
  - Diversidad Lathia: fracción de items nuevos respecto al histórico


### 6.1 Generación de listas top-K
Para cada usuario se puntúan todos los ítems y se excluyen los ya vistos en train.

In [10]:
def get_top_k_lightfm(model, item_features, k=10, n_sample_users=2000, seed=42):
    rng = np.random.default_rng(seed)
    sampled = rng.choice(n_users, size=min(n_sample_users, n_users), replace=False)
    top_k = {}
    for u in sampled:
        # user_ids debe ser array del mismo tamaño que item_ids
        user_ids = np.full(n_items, u, dtype=np.int32)
        scores   = model.predict(user_ids, np.arange(n_items, dtype=np.int32),
                                 user_features=user_feat,
                                 item_features=item_features)
        scores[list(train_seen[u])] = -np.inf
        top_k[u] = np.argsort(scores)[::-1][:k].tolist()
    return top_k

K = 10
topk_base = get_top_k_lightfm(lfm_base, item_feat_base, k=K)
topk_text = get_top_k_lightfm(lfm_text, item_feat_text, k=K)
topk_img = get_top_k_lightfm(lfm_image, item_feat_image, k=K)
topk_full = get_top_k_lightfm(lfm_full, item_feat_full, k=K)


### 6.2 Cálculo de ILS, Novedad y Diversidad Lathia

In [11]:
# Usamos fused_mat como espacio de embeddings para ILS
results = {}
for label, topk in [("base (generos)",           topk_base),
                    ("base + sinopsis",           topk_text),
                     ("base + imagen",           topk_img),
                    ("base + sinopsis + imagen",  topk_full)]:
    ils  = ils_ziegler(topk, fused_mat)
    nov  = novedad_self_info(topk, item_popularity)
    lat  = diversidad_lathia(topk, train_seen, k=K)
    results[label] = {"ILS": ils, "Novedad": nov, "Lathia": lat}
    print(f"[LightFM] {label:30s}  ILS={ils:.4f}  Novedad={nov:.4f}  Lathia={lat:.4f}")
    print(f"           (ILS alto = menos diverso | Novedad alta = items raros | Lathia alta = más nuevos)")

[LightFM] base (generos)                  ILS=22.5176  Novedad=7.9375  Lathia=1.0000
           (ILS alto = menos diverso | Novedad alta = items raros | Lathia alta = más nuevos)
[LightFM] base + sinopsis                 ILS=23.2112  Novedad=7.1727  Lathia=1.0000
           (ILS alto = menos diverso | Novedad alta = items raros | Lathia alta = más nuevos)
[LightFM] base + imagen                   ILS=23.2658  Novedad=7.3817  Lathia=1.0000
           (ILS alto = menos diverso | Novedad alta = items raros | Lathia alta = más nuevos)
[LightFM] base + sinopsis + imagen        ILS=23.3385  Novedad=7.4490  Lathia=1.0000
           (ILS alto = menos diverso | Novedad alta = items raros | Lathia alta = más nuevos)


## 7. Tabla de resultados consolidada

In [12]:
filas = []
for (label, p, r) in [("base (generos)",          p_base, r_base),
                       ("base + sinopsis",         p_text, r_text),
                      ("base + imagen",         p_img, r_img),
                       ("base + sinopsis + imagen",p_full, r_full)]:
    m = results[label]
    filas.append({"Features": label,
                  "P@10": p, "R@10": r,
                  "ILS (↓ mejor)": m["ILS"],
                  "Novedad (↑ mejor)": m["Novedad"],
                  "Lathia (↑ mejor)": m["Lathia"]})

df_res = pd.DataFrame(filas)
print(df_res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

                Features   P@10   R@10  ILS (↓ mejor)  Novedad (↑ mejor)  Lathia (↑ mejor)
          base (generos) 0.0456 0.0179        22.5176             7.9375            1.0000
         base + sinopsis 0.0911 0.0464        23.2112             7.1727            1.0000
           base + imagen 0.0856 0.0428        23.2658             7.3817            1.0000
base + sinopsis + imagen 0.0823 0.0401        23.3385             7.4490            1.0000


## 8. Recomendación real
Codigo base obtenido de IA: https://claude.ai/share/5a9fbec2-9a29-48e1-9a04-9d75da9726b1

In [31]:
items = pd.read_csv("item_table.csv")

idx_to_title  = items.set_index("emb_idx")["clean_title"].to_dict()
idx_to_year   = items.set_index("emb_idx")["year"].to_dict()
idx_to_genres = items.set_index("emb_idx")["genres"].to_dict()

test_csr = test.tocsr()

relevant_test = {}
for u in range(n_users):
    items_en_test = test_csr[u].indices
    ratings_test  = test_csr[u].data
    relevantes    = set(items_en_test[ratings_test >= 4])
    if relevantes:
        relevant_test[u] = relevantes

print(f"Usuarios con al menos 1 relevante en test: {len(relevant_test)}")


def recomendar(model, item_features, user_id, n=10, excluir_vistos=True):
    n_items = item_features.shape[0]
    scores  = model.predict(user_id, np.arange(n_items),
                            user_features=user_feat,
                            item_features=item_features)
    ranking = np.argsort(-scores)

    if excluir_vistos:
        vistos  = train_seen.get(user_id, set())
        ranking = [i for i in ranking if i not in vistos]

    top_n      = ranking[:n]
    relevantes = relevant_test.get(user_id, set())

    resultados = []
    for i in top_n:
        titulo   = idx_to_title.get(i,  f"item_{i}")
        anio     = idx_to_year.get(i,   "?")
        generos  = idx_to_genres.get(i, "?")
        es_hit   = i in relevantes
        resultados.append((titulo, anio, generos, float(scores[i]), es_hit))
    return resultados, relevantes


def comparar_modelos(user_id, n=10):
    modelos = {
        "Base (géneros)"         : (lfm_base,  item_feat_base),
        "Base + sinopsis"        : (lfm_text,  item_feat_text),
        "Base + imagen"          : (lfm_image, item_feat_image),
        "Base + sinopsis + imagen": (lfm_full, item_feat_full),
    }

    for nombre, (modelo, feats) in modelos.items():
        recs, relevantes = recomendar(modelo, feats, user_id, n=n)
        hits = sum(1 for *_, es_hit in recs if es_hit)

        print(f"\n=== {nombre} ===")
        print(f"  Relevantes en test: {len(relevantes)} | "
              f"Hits en top-{n}: {hits}/{n} "
              f"(Precision@{n}={hits/n:.2f})")
        print(f"  {'Título':<45} {'Año':<6} {'Géneros':<35} {'Hit'}")
        print(f"  {'-'*100}")
        for titulo, anio, generos, score, es_hit in recs:
            hit_str = "RELEVANTE" if es_hit else "NO"
            print(f"  {titulo:<45} {str(anio):<6} {str(generos):<35} {hit_str}")


comparar_modelos(user_id=9, n=10)

Usuarios con al menos 1 relevante en test: 5977

=== Base (géneros) ===
  Relevantes en test: 62 | Hits en top-10: 1/10 (Precision@10=0.10)
  Título                                        Año    Géneros                             Hit
  ----------------------------------------------------------------------------------------------------
  Kim                                           1950   Children's|Drama                    NO
  Suture                                        1993   Film-Noir|Thriller                  NO
  Yankee Zulu                                   1994   Comedy|Drama                        NO
  Terminal Velocity                             1994   Action                              NO
  Full Tilt Boogie                              1997   Documentary                         NO
  Who's Afraid of Virginia Woolf?               1966   Drama                               NO
  Indian Summer (a.k.a. Alive & Kicking)        1996   Comedy|Drama                        NO
  Ho